# SHIA-RAG — QuALITY Long-Document Benchmark (Colab)

QuALITY articles are ~4,000–6,000-word **narrative** passages, far larger than the retrieval budget, so the method
that selects the *right* passages wins. Note: QuALITY has no section hierarchy, so this benchmark does **not** exercise
SHIA-RAG's structural advantage (that is what the QASPER benchmark is for) — but it tests the retrieval-efficiency fix.

Compares **SHIA-RAG+** (SHIA selection + query-rerank + tight budget — the fix for 'more context hurt the reader')
and **SHIA-RAG** (base, ablation) against **RAPTOR, TreeRAG, GraphRAG, HiChunk, DTE-RAG** — all answered by the
**same GPU model** (Qwen2.5-3B-Instruct), every method capped to the **same word budget**. Reports multiple-choice
**Accuracy** and **average context size**.

**Step 0:** Runtime → Change runtime type → **T4 GPU** → Save. Run cells top to bottom; upload **shia_core_src.zip** at Cell 2.

In [ ]:
!pip -q install transformers accelerate scikit-learn datasets pydantic numpy

In [ ]:
# Cell 2 — upload your SHIA-RAG code (shia_core_src.zip)
from google.colab import files
import zipfile, os, sys
print("Choose shia_core_src.zip ...")
up = files.upload(); z = list(up)[0]
os.makedirs("shia_core", exist_ok=True)
with zipfile.ZipFile(z) as f: f.extractall("shia_core")
os.makedirs("stubs", exist_ok=True); open("stubs/pymupdf.py","w").write("# stub")
sys.path.insert(0,"stubs"); sys.path.insert(0,"shia_core")
print("SHIA-RAG code ready:", os.listdir("shia_core"))

In [ ]:
# Cell 3 — load QuALITY (long-document multiple-choice QA)
from datasets import load_dataset
N = 100  # firmed-up sample (was 40)
def _parse(ex):
    art = ex.get("article") or ex.get("context") or ex.get("passage") or ""
    q   = ex.get("question","")
    opts= ex.get("options") or ex.get("choices") or []
    gold= ex.get("gold_label", ex.get("answer", ex.get("label")))
    if isinstance(gold,str) and gold.strip() and gold.strip()[0].upper() in "ABCDE":
        gi = "ABCDE".index(gold.strip()[0].upper())
    elif isinstance(gold,int):
        gi = gold-1 if gold>=1 and gold>len(opts)-1 else (gold-1 if gold==len(opts) else gold if gold<len(opts) else gold-1)
    else: gi = gold
    return {"article":art,"q":q,"options":list(opts),"gold":gi}
items=[]
try:
    ds = load_dataset("emozilla/quality", split="validation")
    for ex in ds.select(range(min(N,len(ds)))):
        it=_parse(ex)
        if it["article"] and it["options"] and isinstance(it["gold"],int): items.append(it)
    print("loaded from emozilla/quality")
except Exception as e:
    print("HF load failed:", e, "\n>>> Tell Claude and we'll switch to an uploaded QuALITY file.")
print(len(items), "questions;  example gold index:", items[0]["gold"] if items else None,
      "| #options:", len(items[0]["options"]) if items else None,
      "| article words:", len(items[0]["article"].split()) if items else None)

In [ ]:
# Cell 4 — GPU reader (same model answers every method)
import torch, re
from transformers import AutoModelForCausalLM, AutoTokenizer
MID="Qwen/Qwen2.5-3B-Instruct"
tok=AutoTokenizer.from_pretrained(MID)
model=AutoModelForCausalLM.from_pretrained(MID, torch_dtype=torch.float16, device_map="auto")
L=["A","B","C","D","E"]
def answer_mc(question, options, context):
    opts="\n".join(f"{L[i]}) {o}" for i,o in enumerate(options))
    prompt=("Read the context and answer the multiple-choice question. "
            "Reply with ONLY the letter of the correct option.\n\n"
            f"Context:\n{context}\n\nQuestion: {question}\nOptions:\n{opts}\nEnd with: ANSWER: <letter>")
    msgs=[{"role":"user","content":prompt}]
    text=tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp=tok(text, return_tensors="pt").to(model.device)
    out=model.generate(**inp, max_new_tokens=16, do_sample=False)
    r=tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
    m=re.findall(r"(?i)answer\s*:\s*([A-E])", r) or re.findall(r"\b([A-E])\b", r)
    return L.index(m[-1].upper()) if m else -1
print("reader OK ->", answer_mc("Capital?", ["Paris","Rome","Bonn","Oslo"], "France's capital is Paris."))

In [ ]:
# Cell 5 — long-document retrievers (equal word budget for all)
import sys, re
import logging; logging.getLogger().setLevel(logging.CRITICAL)
from collections import defaultdict
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from src.pipeline import SHIARAGPipeline
from src.layer0_data_model.schemas import DocumentMimeType

BUDGET=512     # WORDS — binding, because a QuALITY article is ~4000-6000 words
SHIA_KEEP=280  # SHIA-RAG+ voluntarily keeps fewer, most-relevant words (its efficient point)

def chunks(article, size=90):
    w=article.split(); return [" ".join(w[i:i+size]) for i in range(0,len(w),size)]
def q_of(item): return item["q"]+" "+" ".join(item["options"])
def cap(text, n=BUDGET): return " ".join(text.split()[:n])

def flat_retrieve(item):
    C=chunks(item["article"])
    v=TfidfVectorizer().fit(C+[q_of(item)])
    sim=cosine_similarity(v.transform([q_of(item)]),v.transform(C))[0]
    out,u=[],0
    for i in sim.argsort()[::-1]:
        w=len(C[i].split())
        if u+w>BUDGET: break
        out.append(C[i]); u+=w
    return "\n".join(out)

def raptor_retrieve(item):
    C=chunks(item["article"])
    v=TfidfVectorizer().fit(C+[q_of(item)])
    Md=v.transform(C).toarray(); qv=v.transform([q_of(item)]).toarray()[0]
    try:
        km=KMeans(n_clusters=min(max(2,int(len(C)**0.5)),len(C)),n_init=5,random_state=0).fit(Md)
        lab=list(km.labels_); ce=km.cluster_centers_
    except Exception:
        lab=[0]*len(C); ce=np.zeros((1,Md.shape[1]))
    ps=(Md@qv)/((np.linalg.norm(Md,axis=1)*np.linalg.norm(qv))+1e-9)
    cs=(ce@qv)/((np.linalg.norm(ce,axis=1)*np.linalg.norm(qv))+1e-9)
    out,u=[],0
    for i in sorted(range(len(C)),key=lambda i:-(ps[i]+0.3*cs[lab[i]])):
        w=len(C[i].split())
        if u+w>BUDGET: break
        out.append(C[i]); u+=w
    return "\n".join(out)

def treerag_retrieve(item):   # structural locality: top chunks + their neighbours
    C=chunks(item["article"])
    v=TfidfVectorizer().fit(C+[q_of(item)])
    sim=cosine_similarity(v.transform([q_of(item)]),v.transform(C))[0]
    order=list(sim.argsort()[::-1]); keep=set(); seq=[]
    for i in order:
        for j in (i-1,i,i+1):
            if 0<=j<len(C) and j not in keep: keep.add(j); seq.append(j)
        if sum(len(C[j].split()) for j in seq)>BUDGET: break
    out,u=[],0
    for j in sorted(seq):
        w=len(C[j].split())
        if u+w>BUDGET: break
        out.append(C[j]); u+=w
    return "\n".join(out)

def _ents(t): return set(re.findall(r"\b[A-Z][a-zA-Z]+(?:\s+[A-Z][a-zA-Z]+)*\b",t))
def graphrag_retrieve(item):
    C=chunks(item["article"]); qe=_ents(q_of(item)); pe=[_ents(c) for c in C]
    sc=[len(qe&e) for e in pe]; order=sorted(range(len(C)),key=lambda i:-sc[i])
    out,u,pk=[],0,set()
    for i in order:
        if sc[i]==0 and pk: break
        w=len(C[i].split())
        if u+w>BUDGET: break
        out.append(C[i]); u+=w; pk.add(i)
    if pk:
        seed=set().union(*[pe[i] for i in pk])
        for i in order:
            if i in pk: continue
            if pe[i]&seed:
                w=len(C[i].split())
                if u+w>BUDGET: break
                out.append(C[i]); u+=w
    return "\n".join(out) if out else flat_retrieve(item)

def hichunk_retrieve(item):   # sentence leaves + auto-merge to chunk
    sents=re.split(r"(?<=[.!?])\s+",item["article"]); 
    C=chunks(item["article"])
    # map sentence -> chunk index by cumulative words
    v=TfidfVectorizer().fit(sents+[q_of(item)])
    sim=cosine_similarity(v.transform([q_of(item)]),v.transform(sents))[0]
    out,u=[],0
    for j in sim.argsort()[::-1]:
        w=len(sents[j].split())
        if u+w>BUDGET: break
        out.append(sents[j]); u+=w
    return "\n".join(out)

def dte_retrieve(item):
    C=chunks(item["article"]); v=TfidfVectorizer().fit(C+[q_of(item)]); Mp=v.transform(C)
    sem=cosine_similarity(v.transform([q_of(item)]),Mp)[0]
    qt=set(q_of(item).lower().split())
    lex=np.array([len(qt&set(c.lower().split()))/((len(qt|set(c.lower().split())))or 1) for c in C])
    lex=lex/(lex.max()+1e-9); base=0.5*sem+0.5*lex; Md=Mp.toarray()
    sel,u,cand=[],0,list(range(len(C)))
    while cand:
        if not sel: i=max(cand,key=lambda i:base[i])
        else:
            i=max(cand,key=lambda i:0.7*base[i]-0.3*max((Md[i]@Md[j])/((np.linalg.norm(Md[i])*np.linalg.norm(Md[j]))+1e-9) for j in sel))
        w=len(C[i].split())
        if u+w>BUDGET: break
        sel.append(i); u+=w; cand.remove(i)
    return "\n".join(C[i] for i in sel)

_QP_PIPE={}; _QP_CACHE={}
def _build_q(item):
    """Build+cache the SHIA pipeline for a narrative article (sentence blocks)."""
    key=hash(item["article"])
    if key in _QP_PIPE: return _QP_PIPE[key]
    p=SHIARAGPipeline(); doc=p.register_document("q",DocumentMimeType.PLAIN,pages=1)
    sents=re.split(r"(?<=[.!?])\s+",item["article"]); o=1
    for s in sents:
        if s.strip(): p.add_text_block(doc.doc_id,s.strip(),o,section_path=f"{o}.0"); o+=1
    for node,par in p.concept_extractor.extract_from_document(doc,list(p.blocks.values())):
        p.add_concept_node(node,candidate_parents=par)
    p.run_confidence_propagation()
    try: p.run_crosslink_discovery()
    except Exception: pass
    p.validate_invariants()
    _QP_PIPE[key]=p; return p

def _shia_blocks(item, mode):
    """Selected block texts. mode='leaf' (leaf-anchored engine) or 'concept' (original)."""
    ck=(hash(item["article"]), q_of(item), mode)
    if ck in _QP_CACHE: return _QP_CACHE[ck]
    p=_build_q(item)
    rr=(p.retrieve_leaf_anchored(q_of(item),token_budget=BUDGET) if mode=="leaf"
        else p.retrieve(q_of(item),token_budget=900))
    seen,txt=set(),[]
    for nid in rr.selected_node_ids:
        for bid in getattr(p.nodes[nid],"source_block_ids",[]):
            b=p.blocks.get(bid)
            if b and b.text_content not in seen: seen.add(b.text_content); txt.append(b.text_content)
    _QP_CACHE[ck]=txt; return txt

def _packed(txt):
    out,u=[],0
    for t in txt:
        w=len(t.split())
        if u+w>BUDGET: break
        out.append(t); u+=w
    return "\n".join(out)

def shia_retrieve(item):          # SHIA-RAG (leaf-anchored precedence — method of record)
    txt=_shia_blocks(item, "leaf")
    return _packed(txt) if txt else flat_retrieve(item)

def shia_concept_retrieve(item):  # SHIA-RAG (concept) — original node-scoring engine (ablation)
    txt=_shia_blocks(item, "concept")
    return _packed(txt) if txt else flat_retrieve(item)

# Flat RAG dropped from the headline table (per project decision — no tree structure);
# the five research-paper baselines below are the comparison set.
RETR={"SHIA-RAG":shia_retrieve,"SHIA-RAG(concept)":shia_concept_retrieve,"RAPTOR":raptor_retrieve,
      "TreeRAG":treerag_retrieve,"GraphRAG":graphrag_retrieve,"HiChunk":hichunk_retrieve,
      "DTE-RAG":dte_retrieve}


print("retrievers ready:", list(RETR))

In [ ]:
# Cell 6 — run and print the QuALITY comparison
import time
M=list(RETR); acc={m:0 for m in M}; cw={m:[] for m in M}; n=0; t0=time.time()
rows=[]
for i,it in enumerate(items,1):
    row={"q":it["q"],"gold":it["gold"]}
    for m in M:
        ctx=cap(RETR[m](it))
        cw[m].append(len(ctx.split()))
        pred=answer_mc(it["q"], it["options"], ctx)
        if pred==it["gold"]: acc[m]+=1
        row[m]=pred
    rows.append(row); n+=1
    if i%5==0: print(f"  {i}/{len(items)} done ({time.time()-t0:.0f}s)")
import json; json.dump({"rows":rows}, open("results_quality.json","w"), indent=2)
order=sorted(M, key=lambda m:-acc[m]/n)
print("\n===== QuALITY RESULTS  ·  reader = Qwen2.5-3B-Instruct (GPU)  ·  n =", n, "=====")
print(f"{'Method':<12}{'Accuracy %':>12}{'AvgCtxWords':>13}")
for m in order: print(f"{m:<12}{100*acc[m]/n:>12.1f}{sum(cw[m])/len(cw[m]):>13.0f}")
print("\nSame reader, same word budget for all — differences are purely retrieval.")